# **Практическое задание №1**

Установка необходимых пакетов:

In [ ]:
!pip install -q tqdm
!pip install --upgrade --no-cache-dir gdown

Монтирование Вашего Google Drive к текущему окружению:

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Константы, которые пригодятся в коде далее, и ссылки (gdrive идентификаторы) на предоставляемые наборы данных:

In [ ]:
EVALUATE_ONLY = False
TEST_ON_LARGE_DATASET = True
TISSUE_CLASSES = ('ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM')
NUM_CLASSES = len(TISSUE_CLASSES)
DATASETS_LINKS = {
    'train': '1XtQzVQ5XbrfxpLHJuL0XBGJ5U7CS-cLi',
    'train_small': '1qd45xXfDwdZjktLFwQb-et-mAaFeCzOR',
    'train_tiny': '1I-2ZOuXLd4QwhZQQltp817Kn3J0Xgbui',
    'test': '1RfPou3pFKpuHDJZ-D9XDFzgvwpUBFlDr',
    'test_small': '1wbRsog0n7uGlHIPGLhyN-PMeT2kdQ2lI',
    'test_tiny': '1viiB0s041CNsAK4itvX8PnYthJ-MDnQc'
}


Подкачка всех .npz датасетов локально на диск

In [ ]:
# for name in DATASETS_LINKS:
#     url = f"https://drive.google.com/uc?export=download&confirm=pbef&id={DATASETS_LINKS[name]}"
#     output = f'/content/drive/MyDrive/{name}.npz'
#     gdown.download(url, output, quiet=False)

Импорт необходимых зависимостей:

In [ ]:
import random
from pathlib import Path

import gdown
import torch
import IPython.display
import numpy as np
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
from torchvision.transforms import v2
from torch.optim.lr_scheduler import OneCycleLR
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

%load_ext tensorboard
from torch.utils.tensorboard import SummaryWriter

Зафиксируем `random_seed=42` для воспроизводимости

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything()

---
### Класс Dataset

Предназначен для работы с наборами данных, обеспечивает чтение изображений и соответствующих меток, а также формирование пакетов (батчей).

In [ ]:
class Dataset:

    def __init__(self, name):
        self.name = name
        self.is_loaded = False
        url = f"https://drive.google.com/uc?export=download&confirm=pbef&id={DATASETS_LINKS[name]}"

        # output = f'{name}.npz'
        # gdown.download(url, output, quiet=False)
        output = f'/content/drive/MyDrive/{name}.npz'
        gdown.download(url, output, quiet=False, skip_download=True)

        print(f'Loading dataset {self.name} from npz.')

        # np_obj = np.load(f'{name}.npz')
        np_obj = np.load(output)

        self.images = np_obj['data']
        self.labels = np_obj['labels']
        self.n_files = self.images.shape[0]
        self.is_loaded = True
        print(f'Done. Dataset {name} consists of {self.n_files} images.')

    def image(self, i):
        # read i-th image in dataset and return it as numpy array
        if self.is_loaded:
            return self.images[i, :, :, :]

    def images_seq(self, n=None):
        # sequential access to images inside dataset (is needed for testing)
        for i in range(self.n_files if not n else n):
            yield self.image(i)

    def random_image_with_label(self):
        # get random image with label from dataset
        i = np.random.randint(self.n_files)
        return self.image(i), self.labels[i]

    def random_batch_with_labels(self, n):
        # create random batch of images with labels (is needed for training)
        indices = np.random.choice(self.n_files, n)
        imgs = [self.image(i) for i in indices]
        labels = np.array([self.labels[i] for i in indices])
        return np.stack(imgs), labels

    def image_with_label(self, i: int):
        # return i-th image with label from dataset
        return self.image(i), self.labels[i]


### Пример использвания класса Dataset
Загрузим обучающий набор данных, получим произвольное изображение с меткой. После чего визуализируем изображение, выведем метку. В будущем, этот кусок кода можно закомментировать или убрать.

In [ ]:
# d_train_tiny = Dataset('train_tiny')

# img, lbl = d_train_tiny.random_image_with_label()
# print()
# print(f'Got numpy array of shape {img.shape}, and label with code {lbl}.')
# print(f'Label code corresponds to {TISSUE_CLASSES[lbl]} class.')

# pil_img = Image.fromarray(img)
# IPython.display.display(pil_img)

### Обёртка над Dataset для использования с PyTorch

In [ ]:
try:
    import torch
    import torchvision.transforms as T
    from PIL import Image
    from torch.utils.data import DataLoader
    from torch.utils.data import Dataset as TorchDataset

    class HistologyTorchDataset(TorchDataset):
        """
        Обёртка над Dataset для использования с PyTorch.

        base_dataset: экземпляр Dataset('train'), Dataset('train_small'), etc.
        transform:    функция/объект, преобразующий изображение (PIL.Image -> torch.Tensor).

        """
        def __init__(self, base_dataset, transform=None):
            self.base = base_dataset
            # Минимальный transform по умолчанию:
            # np.uint8 [0, 255] -> float32 [0.0, 1.0]
            self.transform = transform or T.ToTensor()

        def __len__(self):
            # Размер датасета
            return len(self.base.images)

        def __getitem__(self, idx):
            """
            Возвращает (image_tensor, label) для PyTorch.
            image_tensor: torch.Tensor формы [3, H, W]
            label: int
            """
            img, label = self.base.image_with_label(idx)  # img: np.ndarray (H, W, 3)
            img = Image.fromarray(img)                    # в PIL.Image
            img = self.transform(img)                     # в torch.Tensor
            return img, int(label)

except ImportError:
    HistologyTorchDataset = None
    print("PyTorch / torchvision не найдены. Обёртка HistologyTorchDataset недоступна.")


### Пример использвания класса HistologyTorchDataset

In [ ]:
# if "HistologyTorchDataset" not in globals() or HistologyTorchDataset is None:
#     print("PyTorch не установлен или обёртка недоступна — пример пропущен.")
# else:
#     print("Пример использования PyTorch-обёртки над Dataset")

#     base_train = Dataset('train_tiny')

#     # Создаём PyTorch-совместимый датасет
#     train_ds = HistologyTorchDataset(base_train)

#     # DataLoader автоматически создаёт батчи и перемешивает данные
#     from torch.utils.data import DataLoader
#     train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

#     # Берём один батч и выводим информацию
#     images_batch, labels_batch = next(iter(train_loader))

#     print("Форма батча изображений:", tuple(images_batch.shape))  # [batch, 3, 224, 224]
#     print("Форма батча меток:", tuple(labels_batch.shape))        # [batch]
#     print("Пример меток:", labels_batch[:10].tolist())

#     print("Тип images_batch:", type(images_batch))
#     print("Тип labels_batch:", type(labels_batch))


---
### Класс Metrics

Реализует метрики точности, используемые для оценивания модели:
1. точность,
2. сбалансированную точность.

In [ ]:
class Metrics:

    @staticmethod
    def accuracy(gt: list[int], pred: list[int]):
        assert len(gt) == len(pred), 'gt and prediction should be of equal length'
        return sum(int(i[0] == i[1]) for i in zip(gt, pred)) / len(gt)

    @staticmethod
    def accuracy_balanced(gt: list[int], pred: list[int]):
        return balanced_accuracy_score(gt, pred)

    @staticmethod
    def print_all(gt: list[int], pred: list[int], info: str):
        print(f'metrics for {info}:')
        print(f'\t accuracy {Metrics.accuracy(gt, pred):.4f}:')
        print(f'\t balanced accuracy {Metrics.accuracy_balanced(gt, pred):.4f}:')

---
### Класс Model

Класс, хранящий в себе всю информацию о модели.

Вам необходимо реализовать методы save, load для сохранения и загрузки модели. Особенно актуально это будет во время тестирования на дополнительных наборах данных.


> *Пожалуйста, убедитесь, что сохранение и загрузка модели работает корректно. Для этого обучите модель, протестируйте, сохраните ее в файл, перезапустите среду выполнения, загрузите обученную модель из файла, вновь протестируйте ее на тестовой выборке и убедитесь в том, что получаемые метрики совпадают с полученными для тестовой выборки ранее.*


Также, Вы можете реализовать дополнительные функции, такие как:
1. валидацию модели на части обучающей выборки;
2. использование кроссвалидации;
3. автоматическое сохранение модели при обучении;
4. загрузку модели с какой-то конкретной итерации обучения (если используется итеративное обучение);
5. вывод различных показателей в процессе обучения (например, значение функции потерь на каждой эпохе);
6. построение графиков, визуализирующих процесс обучения (например, график зависимости функции потерь от номера эпохи обучения);
7. автоматическое тестирование на тестовом наборе/наборах данных после каждой эпохи обучения (при использовании итеративного обучения);
8. автоматический выбор гиперпараметров модели во время обучения;
9. сохранение и визуализацию результатов тестирования;
10. Использование аугментации и других способов синтетического расширения набора данных (дополнительным плюсом будет обоснование необходимости и обоснование выбора конкретных типов аугментации)
11. и т.д.

Полный список опций и дополнений приведен в презентации с описанием задания.

При реализации дополнительных функций допускается добавление параметров в существующие методы и добавление новых методов в класс модели.


In [ ]:
class Model:

    def __init__(self, model, num_classes, writer, device):
        self.model = model
        self.num_classes = num_classes
        self.writer = writer  # Для tensorboard
        self.device = device
        self.checkpoint = None

    def save(self, name: str):
        # todo: сохраните веса вашей модели.
        # Пример уже рабочий: он пишет npz на Google Drive, а load читает тот же файл.
        path = f'/content/drive/MyDrive/{name}.npz'
        data = np.asarray(
            self.weights if getattr(self, 'weights', None) is not None
            else np.array([1, 2, 3, 4, 5], dtype=np.float32)
        )
        np.savez(path, data=data)
        print(f'Модель сохранена в {path}.')
        print('Чтобы сдать файл ссылкой, откройте к нему доступ и вставьте id в load().')

    def load(self, name: str):
        # todo: если сдаёте модель ссылкой, замените None на id вашего файла.
        # Пока id не указан, читается файл, который save записал на примонтированный Drive.
        name_to_id_dict = {
            name: None,
        }
        file_id = name_to_id_dict.get(name)
        output = f'{name}.npz'
        if file_id:
            gdown.download(id=file_id, output=output, quiet=False)
            path = output
        else:
            path = f'/content/drive/MyDrive/{output}'
            if not Path(path).exists():
                raise FileNotFoundError(
                    f'Не найден {path}. Вызовите save("{name}") при смонтированном Google Drive '
                    f'или укажите id файла в name_to_id_dict внутри load().'
                )
        np_obj = np.load(path)
        self.weights = np_obj['data']
        print(self.weights)

    def train(self, num_epochs, dataloaders, optimizer, loss_func, scheduler=None, PATH='best_model.pth'):
        info = {
            "valid_loss": [],
            "train_acc": [],
            "valid_acc": [],
            "train_acc_balanced": [],
            "valid_acc_balanced": []
        }
        best_acc = 0.0
        best_acc_balanced = 0.0
        PATIENCE = 3
        patience_counter = 0

        for epoch in range(num_epochs):
            print(f"\nEpoch: {epoch+1}")

            train_preds, train_targets = self._fit_one_epoch(
                train_dataloader=dataloaders['train'],
                optimizer=optimizer,
                loss_func=loss_func,
                scheduler=scheduler
            )
            val_preds, val_targets, val_loss = self._eval_one_epoch(
                val_dataloader=dataloaders['valid'],
                loss_func=loss_func,
            )

            train_acc = Metrics.accuracy(train_targets, train_preds)
            print(f"\n{train_acc = :.4f}")
            valid_acc = Metrics.accuracy(val_targets, val_preds)
            print(f"{valid_acc = :.4f}")

            # Выводим значения
            train_acc_balanced = Metrics.accuracy_balanced(train_targets, train_preds)
            print(f"{train_acc_balanced = :.4f}")
            valid_acc_balanced = Metrics.accuracy_balanced(val_targets, val_preds)
            print(f"{valid_acc_balanced = :.4f}\n")

            if valid_acc > best_acc + 1e-4 and valid_acc_balanced > best_acc_balanced + 1e-4:
                patience_counter = 0
                best_acc = valid_acc
                best_acc_balanced = valid_acc_balanced

                best_checkpoint = {
                    'model_state': model.state_dict(),
                    'arch': 'efficient_net_v2_s',
                    'num_classes': self.num_classes,

                    'optimizer_arch': 'AdamW',
                    'optimizer_state': optimizer.state_dict(),
                    'scheduler_arch': 'OneCycleLR',
                    'scheduler_state': scheduler.state_dict(),

                    'epoch': epoch,
                }
                torch.save(best_checkpoint, PATH)
                print(f"  -> new best accuracy={best_acc:.4f}, saved to {PATH}")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print("Early Stopping! Модель переобучается!")

            # Добавляем информации для tensorboard
            self.writer.add_scalar('loss/valid', val_loss, epoch)
            self.writer.add_scalar('acc/train', train_acc, epoch)
            self.writer.add_scalar('acc/valid', valid_acc, epoch)
            self.writer.add_scalar('acc_balanced/train', train_acc_balanced, epoch)
            self.writer.add_scalar('acc_balanced/valid', valid_acc_balanced, epoch)

            info['valid_loss'].append(val_loss)
            info['train_acc'].append(train_acc)
            info['valid_acc'].append(valid_acc)
            info['train_acc_balanced'].append(train_acc_balanced)
            info['valid_acc_balanced'].append(valid_acc_balanced)
        return info

    def test_on_dataset(self, dataset: HistologyTorchDataset, limit=None):
        # you can upgrade this code if you want to speed up testing using batches
        # TODO: Исправить код так, чтобы он работал именно для TorchDataset
        predictions = []
        n = dataset.n_files if not limit else int(dataset.n_files * limit)
        for img in tqdm(dataset.images_seq(n), total=n):
            predictions.append(self.test_on_image(img, 2026))
        return predictions

    def test_on_image(self, img: np.ndarray, submission_year: int):
        # todo: replace this code
        prediction = int(np.random.randint(9))
        return prediction

    def _fit_one_epoch(self, train_dataloader, optimizer, loss_func, scheduler=None):
        self.model.train()

        all_preds = []
        all_targets = []

        running_loss = 0.0
        total = 0

        for X_batch, y_batch in tqdm(train_dataloader, desc='Training...'):
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)

            optimizer.zero_grad()

            outp = model(X_batch)
            loss = loss_func(outp, y_batch)

            loss.backward()
            optimizer.step()
            if scheduler:
                scheduler.step()

            preds = outp.argmax(dim=1)

            running_loss += loss.item() * X_batch.size(0)
            total += y_batch.size(0)

            all_preds.append(preds.detach().cpu())
            all_targets.append(y_batch.detach().cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        return all_preds, all_targets

    def _eval_one_epoch(self, val_dataloader, loss_func):
        self.model.eval()

        all_preds = []
        all_targets = []

        running_loss = 0.0
        total = 0

        for X_batch, y_batch in tqdm(val_dataloader, desc='Validation...'):
            with torch.no_grad():
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                outp = model(X_batch)
                loss = loss_func(outp, y_batch)

                preds = outp.argmax(dim=1)

                running_loss += loss.item() * X_batch.size(0)
                total += y_batch.size(0)

                all_preds.append(preds.detach().cpu())
                all_targets.append(y_batch.detach().cpu())

        epoch_loss = running_loss / total
        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        return all_preds, all_targets, epoch_loss

---
### Классификация изображений

Используя введенные выше классы можем перейти уже непосредственно к обучению модели классификации изображений. Пример общего пайплайна решения задачи приведен ниже. Вы можете его расширять и улучшать. В данном примере используются наборы данных 'train_small' и 'test_small'.

In [ ]:
d_train = HistologyTorchDataset('train_small')
d_test = HistologyTorchDataset('test_small')

In [ ]:
train, val = train_test_split(d_train, test_size=0.2)

In [ ]:
batch_size = 128

In [ ]:
train_dataloader = DataLoader(train, batch_size, shuffle=True, num_workers=2)
valid_dataloader = DataLoader(val, batch_size, shuffle=False, num_workers=2)
test_dataloader = DataLoader(d_test, batch_size, shuffle=False)

dataloaders = {
    'train': train_dataloader,
    'valid': valid_dataloader,
    'test': test_dataloader
}

In [ ]:
model = Model()
if not EVALUATE_ONLY:
    model.train(d_train)
    model.save('best')
else:
    #todo: your link goes here
    model.load('best')

Пример тестирования модели на части набора данных:

In [ ]:
# evaluating model on 10% of test dataset
pred_1 = model.test_on_dataset(d_test, limit=0.1)
Metrics.print_all(d_test.labels[:len(pred_1)], pred_1, '10% of test')

Пример тестирования модели на полном наборе данных:

In [ ]:
# evaluating model on full test dataset (may take time)
if TEST_ON_LARGE_DATASET:
    pred_2 = model.test_on_dataset(d_test)
    Metrics.print_all(d_test.labels, pred_2, 'test')

Результат работы пайплайна обучения и тестирования выше тоже будет оцениваться. Поэтому не забудьте присылать на проверку ноутбук с выполнеными ячейками кода с демонстрациями метрик обучения, графиками и т.п. В этом пайплайне Вам необходимо продемонстрировать работу всех реализованных дополнений, улучшений и т.п.

<font color="red">
Настоятельно рекомендуется после получения пайплайна с полными результатами обучения экспортировать ноутбук в pdf (файл -> печать) и прислать этот pdf вместе с самим ноутбуком.
</font>

### Тестирование модели на других наборах данных

Ваша модель должна поддерживать тестирование на других наборах данных. Для удобства, Вам предоставляется набор данных test_tiny, который представляет собой малую часть (2% изображений) набора test. Ниже приведен фрагмент кода, который будет осуществлять тестирование для оценивания Вашей модели на дополнительных тестовых наборах данных.

<font color="red">
Прежде чем отсылать задание на проверку, убедитесь в работоспособности фрагмента кода ниже.
</font>


In [ ]:
final_model = Model()
final_model.load('best')
d_test_tiny = Dataset('test_tiny')
pred = final_model.test_on_dataset(d_test_tiny)
Metrics.print_all(d_test_tiny.labels, pred, 'test-tiny')

Google Drive пока не отключаем: примеры архивов ниже пишут файлы на Диск. Отключение стоит в последней ячейке.


In [ ]:
# Диск остаётся примонтированным до последней ячейки.


---
# Дополнительные "полезности"

Ниже приведены примеры использования различных функций и библиотек, которые могут быть полезны при выполнении данного практического задания.

### Измерение времени работы кода

Измерять время работы какой-либо функции можно легко и непринужденно при помощи функции timeit из соответствующего модуля:

In [ ]:
import timeit


def factorial(n):
    res = 1
    for i in range(1, n + 1):
        res *= i
    return res


def f():
    return factorial(n=1000)

n_runs = 128
print(f'Function f is caluclated {n_runs} times in {timeit.timeit(f, number=n_runs)}s.')

### Scikit-image

Реализовывать различные операции для работы с изображениями можно как самостоятельно, работая с массивами numpy, так и используя специализированные библиотеки, например, scikit-image (https://scikit-image.org/). Ниже приведен пример использования Canny edge detector.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi
from skimage import feature

# Generate noisy image of a square
im = np.zeros((128, 128))
im[32:-32, 32:-32] = 1

im = ndi.rotate(im, 15, mode='constant')
im = ndi.gaussian_filter(im, 4)
im += 0.2 * np.random.random(im.shape)

# Compute the Canny filter for two values of sigma
edges1 = feature.canny(im)
edges2 = feature.canny(im, sigma=3)

# display results
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3, figsize=(8, 3),
                                    sharex=True, sharey=True)

ax1.imshow(im, cmap=plt.cm.gray)
ax1.axis('off')
ax1.set_title('noisy image', fontsize=20)

ax2.imshow(edges1, cmap=plt.cm.gray)
ax2.axis('off')
ax2.set_title(r'Canny filter, $\sigma=1$', fontsize=20)

ax3.imshow(edges2, cmap=plt.cm.gray)
ax3.axis('off')
ax3.set_title(r'Canny filter, $\sigma=3$', fontsize=20)

fig.tight_layout()

plt.show()

<font color="red">
Для эффективной работы с моделями глубокого обучения убедитесь в том, что в текущей среде Google Colab используется аппаратный ускоритель GPU или TPU. Для смены среды выберите "среда выполнения" -> "сменить среду выполнения".
</font>

### PyTorch

Готовой сети и цикла обучения в шаблоне нет: это часть задания. Данные для PyTorch берёт класс `HistologyTorchDataset` выше. Ниже — ссылки на документацию.


### Дополнительные ресурсы по PyTorch

- **Официальные туториалы PyTorch** — https://pytorch.org/tutorials/  
- **“Deep Learning with PyTorch: 60-Minute Blitz”** — https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html  
- **Transfer Learning for Computer Vision** — https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html  
- **PyTorch Get Started (установка)** — https://pytorch.org/get-started/locally/  
- **Dive into Deep Learning (D2L, глава PyTorch)** — https://d2l.ai/chapter_preliminaries/index.html  
- **Fast.ai — Practical Deep Learning for Coders** — https://course.fast.ai/  
- **Learn PyTorch.io (Zero to Mastery)** — https://www.learnpytorch.io/

### Работа с zip архивами в Google Drive

Запаковка и распаковка zip архивов может пригодиться при сохранении и загрузки Вашей модели. Ниже приведен фрагмент кода, иллюстрирующий помещение нескольких файлов в zip архив с последующим чтением файлов из него. Все действия с директориями, файлами и архивами должны осущетвляться с примонтированным Google Drive.


Создадим 2 изображения, поместим их в директорию tmp внутри PROJECT_DIR, запакуем директорию tmp в архив tmp.zip.

In [ ]:
# PROJECT_DIR = "prak_nn_1"
# arr1 = np.random.rand(100, 100, 3) * 255
# arr2 = np.random.rand(100, 100, 3) * 255

# img1 = Image.fromarray(arr1.astype('uint8'))
# img2 = Image.fromarray(arr2.astype('uint8'))

# p = str(Path("/content/drive/MyDrive") / PROJECT_DIR)
# Path(p, "tmp").mkdir(parents=True, exist_ok=True)

# img1.save(str(Path(p) / "tmp" / "img1.png"))
# img2.save(str(Path(p) / "tmp" / "img2.png"))

# %cd $p
# !zip -r "tmp.zip" "tmp"


Распакуем архив tmp.zip в директорию tmp2 в PROJECT_DIR. Теперь внутри директории tmp2 содержится директория tmp, внутри которой находятся 2 изображения.

In [ ]:
# p = str(Path("/content/drive/MyDrive") / PROJECT_DIR)
# %cd $p
# !unzip -uq "tmp.zip" -d "tmp2"
# %cd /content


Отмонтировать Google Drive.


In [ ]:
drive.flush_and_unmount()
